Güvensiz vs Güvenli Login Sistemi

`Güvensiz Login Sistemi`: SQL Injection ve şifreleme eksikliği içerir.
`Güvenli Login Sistemi`: Parametrik sorgular ve `bcrypt` kullanır.

Amaç, güvenli ve güvensiz kod arasındaki farkları göstermek ve güvenli yazılım geliştirme sağlamak.

In [12]:
import sqlite3
import bcrypt


In [16]:
# Eğer eski bir veritabanı varsa bu kodla silebilirsin: !rm users.db

# Veritabanını oluştur
conn = sqlite3.connect("users.db")
cursor = conn.cursor()

# Kullanıcılar tablosu
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    password BLOB NOT NULL
)
""")

# Şifreleri bcrypt ile hashleyerek kayıt ekleyelim
def add_user(username, plain_password):
    hashed_pw = bcrypt.hashpw(plain_password.encode(), bcrypt.gensalt())
    cursor.execute("INSERT OR IGNORE INTO users (username, password) VALUES (?, ?)", (username, hashed_pw))
    conn.commit()

# Örnek kullanıcı
add_user("admin", "admin123")
add_user("test", "test123")
print("Kullanıcılar oluşturuldu.")


Kullanıcılar oluşturuldu.


In [14]:
def insecure_login(username, password):
    # SQL Injection'a açık sorgu ❌
    query = f"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'"
    print(f"Çalıştırılan sorgu: {query}")
    
    cursor.execute(query)
    result = cursor.fetchone()
    
    if result:
        print("✅ Giriş başarılı (GÜVENSİZ)")
    else:
        print("❌ Hatalı giriş")

# Test: Doğru giriş
insecure_login("admin", "admin123")

# Test: SQL Injection ile giriş (şifre gerekmeden)
insecure_login("admin", "' OR '1'='1")


Çalıştırılan sorgu: SELECT * FROM users WHERE username = 'admin' AND password = 'admin123'
❌ Hatalı giriş
Çalıştırılan sorgu: SELECT * FROM users WHERE username = 'admin' AND password = '' OR '1'='1'
✅ Giriş başarılı (GÜVENSİZ)


In [15]:
def secure_login(username, password):
    query = "SELECT password FROM users WHERE username = ?"
    cursor.execute(query, (username,))
    result = cursor.fetchone()

    if result and bcrypt.checkpw(password.encode(), result[0]):
        print("✅ Giriş başarılı (GÜVENLİ)")
    else:
        print("❌ Hatalı giriş")

# Test: Doğru giriş
secure_login("admin", "admin123")

# Test: SQL Injection denemesi başarısız olacak
secure_login("admin", "' OR '1'='1")


✅ Giriş başarılı (GÜVENLİ)
❌ Hatalı giriş


## Karşılaştırma: Güvensiz vs Güvenli Login Sistemi

| Özellik                         | Güvensiz Sistem        | Güvenli Sistem           |
|-------------------------------|-------------------------|---------------------------|
| SQL Injection Koruması        | ❌ Yok                  | ✅ Parametrik Sorgu       |
| Şifreleme (Hashleme)          | ❌ Düz metin            | ✅ bcrypt ile hashleme    |
| Hata Mesajları                | Zayıf / açık veriyor   | Daha kontrollü            |
| Kod Güvenliği                 | Zayıf                   | Güçlü                     |
